# Dataset & DataModule tests
Tests for `DatasetCatCon` and `PT_FT_DataModule` (pretrain + finetune phases).

In [4]:
import os
import torch
from time import time
from omegaconf import OmegaConf

from fm4tag.data import DatasetCatCon, cat_con_collate_fn, PT_FT_DataModule

In [6]:
CONFIG_PATH = '../src/fm4tag/configs/saintV0.yaml'
assert os.path.exists(CONFIG_PATH), f'Config not found: {CONFIG_PATH}'
cfg = OmegaConf.load(CONFIG_PATH)
print(OmegaConf.to_yaml(cfg))

global_object: jets
constituent_objects:
- tracks
variables:
  jets:
    inputs:
    - pt_btagJes
    - eta_btagJes
    label: flavour_label
    unique_labels:
    - 0
    - 1
    - 2
    - 3
  tracks:
    inputs:
      continuous:
      - d0
      - z0SinTheta
      - dphi
      - deta
      - qOverP
      - lifetimeSignedD0Significance
      - lifetimeSignedZ0SinThetaSignificance
      - phiUncertainty
      - thetaUncertainty
      - qOverPUncertainty
      categorical:
      - numberOfPixelHits
      - numberOfSCTHits
      - numberOfInnermostPixelLayerHits
      - numberOfNextToInnermostPixelLayerHits
      - numberOfInnermostPixelLayerSharedHits
      - numberOfInnermostPixelLayerSplitHits
      - numberOfPixelSharedHits
      - numberOfPixelSplitHits
      - numberOfSCTSharedHits
      cat_classes:
        numberOfPixelHits:
        - 0
        - 1
        - 2
        - 3
        - 4
        - 5
        - 6
        - 7
        - 8
        - 9
        - 10
        - 11
        - 

## 1 — DatasetCatCon: single-sample inspection

In [7]:
# Build a dataset kwargs dict exactly as the datamodule does internally
ds_kwargs = dict(
    variables=cfg.variables,
    global_object=cfg.global_object,
    constituent_objects=list(cfg.constituent_objects),
    norm_dict=OmegaConf.to_container(OmegaConf.load(cfg.norm_dict), resolve=True),
    class_dict=OmegaConf.to_container(OmegaConf.load(cfg.class_dict), resolve=True),
)

pretrain_ds = DatasetCatCon(file_path=cfg.pretrain_file, **ds_kwargs)
train_ds = DatasetCatCon(file_path=cfg.train_file, **ds_kwargs)
print(f'pretrain samples : {len(pretrain_ds)}')
print(f'train    samples : {len(train_ds)}')


DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/train/output/pp_output_train.h5
  samples : 300000
pretrain samples : 500000
train    samples : 300000


In [8]:
sample = pretrain_ds[0]

print('Keys:', list(sample.keys()))
print(f'  label  : {sample["label"]}  dtype={sample["label"].dtype}')
print(f'  global : shape={sample["global"].shape}  dtype={sample["global"].dtype}')

for obj, feats in sample['constituents'].items():
    cat = feats['categorical']
    con = feats['continuous']
    val = feats['valid']
    n_valid = val.sum().item()
    print(f'  {obj}:')
    print(f'    categorical : shape={cat.shape}  dtype={cat.dtype}')
    print(f'    continuous  : shape={con.shape}  dtype={con.dtype}')
    print(f'    valid       : shape={val.shape}  n_valid={n_valid}')

Keys: ['label', 'global', 'constituents']
  label  : 1  dtype=torch.int64
  global : shape=torch.Size([2])  dtype=torch.float32
  tracks:
    categorical : shape=torch.Size([40, 9])  dtype=torch.uint8
    continuous  : shape=torch.Size([40, 10])  dtype=torch.float32
    valid       : shape=torch.Size([40])  n_valid=5


In [9]:
from torch.utils.data.dataloader import default_collate

BATCH_SIZE = 8
raw_batch = [pretrain_ds[i] for i in range(BATCH_SIZE)]

custom = cat_con_collate_fn(raw_batch)
default = default_collate(raw_batch)

obj = list(custom['constituents'].keys())[0]

print(f'{"Field":<35}  {"custom":>22}  {"default":>22}')
print('-' * 85)

fields_custom = [
    ('label', custom['label']),
    ('global', custom['global']),
    (f'{obj} / categorical', custom['constituents'][obj]['categorical']),
    (f'{obj} / continuous', custom['constituents'][obj]['continuous']),
    (f'{obj} / valid', custom['constituents'][obj]['valid']),
]

fields_default = [
    ('label', default['label']),
    ('global', default['global']),
    (f'{obj} / categorical', default['constituents'][obj]['categorical']),
    (f'{obj} / continuous', default['constituents'][obj]['continuous']),
    (f'{obj} / valid', default['constituents'][obj]['valid']),
]


for name, c in fields_custom:
    c_info = f'shape={tuple(c.shape)} {c.dtype}'
    print(f'  {name:<33}  {c_info:>22}')

for name, d in fields_default:
    d_info = f'shape={tuple(d.shape)} {d.dtype}'
    print(f'  {name:<33}  {d_info:>22}')

Field                                                custom                 default
-------------------------------------------------------------------------------------
  label                              shape=(8,) torch.int64
  global                             shape=(8, 2) torch.float32
  tracks / categorical               shape=(8, 40, 9) torch.int64
  tracks / continuous                shape=(8, 40, 10) torch.float32
  tracks / valid                     shape=(8, 40) torch.bool
  label                              shape=(8,) torch.int64
  global                             shape=(8, 2) torch.float32
  tracks / categorical               shape=(8, 40, 9) torch.uint8
  tracks / continuous                shape=(8, 40, 10) torch.float32
  tracks / valid                     shape=(8, 40) torch.bool


## 2 — Collate function: batch shapes and dtypes

In [10]:
BATCH_SIZE = 8
raw_batch = [pretrain_ds[i] for i in range(BATCH_SIZE)]
batch = cat_con_collate_fn(raw_batch)

print(f'label  : shape={batch["label"].shape}  dtype={batch["label"].dtype}')
print(f'global : shape={batch["global"].shape}  dtype={batch["global"].dtype}')

for obj, feats in batch['constituents'].items():
    print(f'{obj}:')
    print(
        f'  categorical : shape={feats["categorical"].shape}  dtype={feats["categorical"].dtype}'
    )
    print(
        f'  continuous  : shape={feats["continuous"].shape}   dtype={feats["continuous"].dtype}'
    )
    print(
        f'  valid       : shape={feats["valid"].shape}        dtype={feats["valid"].dtype}'
    )

label  : shape=torch.Size([8])  dtype=torch.int64
global : shape=torch.Size([8, 2])  dtype=torch.float32
tracks:
  categorical : shape=torch.Size([8, 40, 9])  dtype=torch.int64
  continuous  : shape=torch.Size([8, 40, 10])   dtype=torch.float32
  valid       : shape=torch.Size([8, 40])        dtype=torch.bool


## 3 — PT_FT_DataModule: pretrain phase

In [11]:
dm_pretrain = PT_FT_DataModule(cfg, phase='pretrain')
dm_pretrain.setup('fit')

pt_loader = dm_pretrain.train_dataloader()
pt_val = dm_pretrain.val_dataloader()

print(f'pretrain train batches : {len(pt_loader)}')
print(f'pretrain val loader    : {pt_val!r}  ([] means no val file set, expected)')


DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000
pretrain train batches : 488
pretrain val loader    : []  ([] means no val file set, expected)


In [12]:
# Fetch one batch from the pretrain train loader and verify shapes
batch_pt = next(iter(pt_loader))

B = batch_pt['label'].shape[0]
print(f'batch_size : {B}')
print(f'label  : {batch_pt["label"].shape}  dtype={batch_pt["label"].dtype}')
print(f'global : {batch_pt["global"].shape}  dtype={batch_pt["global"].dtype}')

for obj, feats in batch_pt['constituents'].items():
    print(f'{obj}:')
    print(
        f'  categorical : {feats["categorical"].shape}  dtype={feats["categorical"].dtype}'
    )
    print(
        f'  continuous  : {feats["continuous"].shape}   dtype={feats["continuous"].dtype}'
    )
    print(f'  valid       : {feats["valid"].shape}        dtype={feats["valid"].dtype}')

batch_size : 1024
label  : torch.Size([1024])  dtype=torch.int64
global : torch.Size([1024, 2])  dtype=torch.float32
tracks:
  categorical : torch.Size([1024, 40, 9])  dtype=torch.int64
  continuous  : torch.Size([1024, 40, 10])   dtype=torch.float32
  valid       : torch.Size([1024, 40])        dtype=torch.bool


## 4 — PT_FT_DataModule: finetune phase

In [13]:
dm_finetune = PT_FT_DataModule(cfg, phase='finetune')
dm_finetune.setup('fit')
dm_finetune.setup('test')

ft_train = dm_finetune.train_dataloader()
ft_val = dm_finetune.val_dataloader()
ft_test = dm_finetune.test_dataloader()

print(f'finetune train batches : {len(ft_train)}')
print(f'finetune val   batches : {len(ft_val)}')
print(f'finetune test  batches : {len(ft_test)}')


DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/train/output/pp_output_train.h5
  samples : 300000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/test/output/pp_output_test.h5
  samples : 100000
finetune train batches : 292
finetune val   batches : 98
finetune test  batches : 98


In [18]:
# Verify all three splits return consistent shapes
for name, loader in [('train', ft_train), ('val', ft_val), ('test', ft_test)]:
    b = next(iter(loader))
    obj = list(b['constituents'].keys())[0]
    print(
        f'{name:5s}  label={b["label"].shape}  '
        f'global={b["global"].shape}  '
        f'con={b["constituents"][obj]["continuous"].shape}  '
        f'cat={b["constituents"][obj]["categorical"].shape}'
    )

train  label=torch.Size([1024])  global=torch.Size([1024, 2])  con=torch.Size([1024, 40, 10])  cat=torch.Size([1024, 40, 9])
val    label=torch.Size([1024])  global=torch.Size([1024, 2])  con=torch.Size([1024, 40, 10])  cat=torch.Size([1024, 40, 9])
test   label=torch.Size([1024])  global=torch.Size([1024, 2])  con=torch.Size([1024, 40, 10])  cat=torch.Size([1024, 40, 9])


## 5 — Label distribution check

In [19]:
# Accumulate labels from the first 20 finetune train batches
labels_seen = []
for i, b in enumerate(ft_train):
    labels_seen.append(b['label'])
    if i >= 19:
        break

all_labels = torch.cat(labels_seen)
unique, counts = torch.unique(all_labels, return_counts=True)
print('Label distribution (first 20 batches):')
for u, c in zip(unique.tolist(), counts.tolist()):
    print(f'  class {u}: {c}  ({100 * c / len(all_labels):.1f}%)')

Label distribution (first 20 batches):
  class 0: 5044  (24.6%)
  class 1: 5274  (25.8%)
  class 2: 5118  (25.0%)
  class 3: 5044  (24.6%)


## 6 — Throughput benchmark

In [20]:
N_BATCHES = 50

for phase, loader in [('pretrain', pt_loader), ('finetune', ft_train)]:
    t0 = time()
    for i, _ in enumerate(loader):
        if i >= N_BATCHES - 1:
            break
    elapsed = time() - t0
    bs = loader.batch_size
    samples = N_BATCHES * bs
    print(
        f'{phase:9s}  {N_BATCHES} batches × {bs}  →  {elapsed:.2f}s  ({samples / elapsed:,.0f} samples/s)'
    )

pretrain   50 batches × 1024  →  22.23s  (2,304 samples/s)
finetune   50 batches × 1024  →  22.11s  (2,316 samples/s)


In [22]:
for i, b in enumerate(pt_loader):
    if i >= 2:
        break
    print(f'Batch {i}:')
    print(f'  label  : shape={b["label"].shape}  dtype={b["label"].dtype}')
    print(f'  global : shape={b["global"].shape}  dtype={b["global"].dtype}')
    for obj, feats in b['constituents'].items():
        print(f'  {obj}:')
        print(
            f'    categorical : shape={feats["categorical"].shape}  dtype={feats["categorical"].dtype}'
        )
        print(
            f'    continuous  : shape={feats["continuous"].shape}   dtype={feats["continuous"].dtype}'
        )
        print(
            f'    valid       : shape={feats["valid"].shape}        dtype={feats["valid"].dtype}'
        )

Batch 0:
  label  : shape=torch.Size([1024])  dtype=torch.int64
  global : shape=torch.Size([1024, 2])  dtype=torch.float32
  tracks:
    categorical : shape=torch.Size([1024, 40, 9])  dtype=torch.int64
    continuous  : shape=torch.Size([1024, 40, 10])   dtype=torch.float32
    valid       : shape=torch.Size([1024, 40])        dtype=torch.bool
Batch 1:
  label  : shape=torch.Size([1024])  dtype=torch.int64
  global : shape=torch.Size([1024, 2])  dtype=torch.float32
  tracks:
    categorical : shape=torch.Size([1024, 40, 9])  dtype=torch.int64
    continuous  : shape=torch.Size([1024, 40, 10])   dtype=torch.float32
    valid       : shape=torch.Size([1024, 40])        dtype=torch.bool
